# CoppyLora_webUI 環境構築＆実行ノート
このノートでは `CoppyLora_webUI.py` を起動するための環境構築から実行までの手順をまとめています。
Python 3.11 と CUDA 12.1 対応 GPU ドライバがインストールされた環境を想定しています。

In [ ]:
# 依存ライブラリのインストール
!pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!pip install -r sd-scripts/requirements.txt
!pip install xformers==0.0.23.post1 --index-url https://download.pytorch.org/whl/cu121
!pip install -U bitsandbytes
!pip install wandb==0.17.3 gradio==4.44.1 pyinstaller
!pip install onnx==1.15.0 onnxruntime==1.17.1 onnxruntime-gpu==1.17.1!pip install toml


In [ ]:
# sd-scripts のクローン
import os, subprocess
if not os.path.exists('sd-scripts'):
    subprocess.run(['git', 'clone', 'https://github.com/kohya-ss/sd-scripts.git'], check=True)

In [ ]:
# sd-scripts の一部ファイルを修正
import pathlib
lpw = pathlib.Path('sd-scripts/library/lpw_stable_diffusion.py')
sdxl_lpw = pathlib.Path('sd-scripts/library/sdxl_lpw_stable_diffusion.py')
old = 'from diffusers.pipelines.stable_diffusion import StableDiffusionPipelineOutput, StableDiffusionSafetyChecker'
new = ('from diffusers.pipelines.stable_diffusion import StableDiffusionPipelineOutput\n'
       'from diffusers.pipelines.stable_diffusion.safety_checker import StableDiffusionSafetyChecker')
for path in [lpw, sdxl_lpw]:
    if path.exists():
        text = path.read_text()
        if old in text:
            path.write_text(text.replace(old, new))

## モデルのダウンロード
モデルファイルは数 GB あるため、実行環境に応じて以下のコマンドを実行してください。
Windows 環境の場合は `CoppyLora_webUI_DL.cmd` を PowerShell から実行することでまとめて取得できます。
ローカル環境で実行する場合、`models/SDXL` は `/notebooks/shared-storage/models/sdxl` にリンクしておくとディスク容量を節約できます。


In [ ]:
%%bash
# 例: Linux 環境での個別ダウンロード例
# ディレクトリを作成し、必要であれば SDXL 用の共有ストレージへのシンボリックリンクを作成します。
mkdir -p models models/tagger models/LoRA
if [ ! -e models/SDXL ]; then ln -s /notebooks/shared-storage/models/sdxl models/SDXL; fi

# SDXL モデルの取得 (共有ストレージに保存されます)
curl -L https://huggingface.co/cagliostrolab/animagine-xl-3.1/resolve/main/animagine-xl-3.1.safetensors -o models/SDXL/animagine-xl-3.1.safetensors

# Tagger モデルの取得
curl -L https://huggingface.co/SmilingWolf/wd-swinv2-tagger-v3/resolve/main/config.json -o models/tagger/config.json
curl -L https://huggingface.co/SmilingWolf/wd-swinv2-tagger-v3/resolve/main/model.onnx -o models/tagger/model.onnx
curl -L https://huggingface.co/SmilingWolf/wd-swinv2-tagger-v3/resolve/main/selected_tags.csv -o models/tagger/selected_tags.csv
curl -L https://huggingface.co/SmilingWolf/wd-swinv2-tagger-v3/resolve/main/sw_jax_cv_config.json -o models/tagger/sw_jax_cv_config.json

# LoRA モデルの取得
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-boy_cl_am31.safetensors -o models/LoRA/copi-ki-base-boy_cl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-boy_ncl_am31.safetensors -o models/LoRA/copi-ki-base-boy_ncl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-boy_ncnl_am31.safetensors -o models/LoRA/copi-ki-base-boy_ncnl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-boy_cnl_am31.safetensors -o models/LoRA/copi-ki-base-boy_cnl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-girl_cl_am31.safetensors -o models/LoRA/copi-ki-base-girl_cl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-girl_ncl_am31.safetensors -o models/LoRA/copi-ki-base-girl_ncl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-girl_ncnl_am31.safetensors -o models/LoRA/copi-ki-base-girl_ncnl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-girl_cnl_am31.safetensors -o models/LoRA/copi-ki-base-girl_cnl_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-female_p_am31.safetensors -o models/LoRA/copi-ki-base-female_p_am31.safetensors
curl -L https://huggingface.co/tori29umai/mylora_V2/resolve/main/copi-ki-base-male_p_am31.safetensors -o models/LoRA/copi-ki-base-male_p_am31.safetensors


In [ ]:
# Web UI の起動
!python CoppyLora_webUI.py